In [ ]:
%pip install -U datasets huggingface_hub

In [2]:
import pandas as pd
import re

url = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Electronics.jsonl"

df = pd.read_json(url, lines = True, nrows = 50000)  # nrows limits to 50k so it doesn't take forever

print(df.shape)
print(df.columns.tolist())
print(df.head(2))

(50000, 10)
['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
   rating                                    title  \
0       3        Smells like gasoline! Going back!   
1       1  Didn’t work at all lenses loose/broken.   

                                                text  \
0  First & most offensive: they reek of gasoline ...   
1  These didn’t work. Idk if they were damaged in...   

                                              images        asin parent_asin  \
0  [{'small_image_url': 'https://m.media-amazon.c...  B083NRGZMM  B083NRGZMM   
1                                                 []  B07N69T6TM  B07N69T6TM   

                        user_id               timestamp  helpful_vote  \
0  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ 2022-07-18 22:58:37.948             0   
1  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ 2020-06-20 18:42:29.731             0   

   verified_purchase  
0               True  
1               True  


In [16]:
import requests
import json

response = requests.get(meta_url, stream = True)
lines = []
for line in response.iter_lines():
    if len(lines) >= 50000:
        break
    if line:
        lines.append(line)

meta = [json.loads(l) for l in lines]
df_meta = pd.DataFrame(meta)

df_meta['subcategory'] = df_meta['categories'].apply(lambda x: x[1] if isinstance(x, list) and len(x) > 1 else 'Unknown')
df_meta = df_meta[['parent_asin', 'subcategory', 'title']].rename(columns={'title': 'product_name'})

df = df.drop(columns=['subcategory', 'product_name'])
df = df.merge(df_meta, on='parent_asin', how='left')

print(df['subcategory'].value_counts().head(10))
print(f"Still unmatched: {df['subcategory'].isna().sum()}")

subcategory
Computers & Accessories              1584
Unknown                               396
Headphones, Earbuds & Accessories     349
Camera & Photo                        295
Television & Video                    236
Portable Audio & Video                173
Accessories & Supplies                150
Wearable Technology                    87
Home Audio                             62
eBook Readers & Accessories            36
Name: count, dtype: int64
Still unmatched: 36123


In [17]:
df = df[['asin', 'parent_asin', 'rating', 'title', 'text', 'verified_purchase', 'timestamp']]

In [18]:
df = df.sort_values('timestamp').reset_index(drop = True)

In [19]:
def clean_text(text):
    if not isinstance(text, str):
        return None
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text if len(text.split()) >= 10 else None

df['text'] = df['text'].apply(clean_text)
df['title'] = df['title'].apply(clean_text)

In [20]:
df = df.dropna(subset = ['text'])

In [21]:
df = df.reset_index(drop = True)

In [22]:
df.to_csv('electronics_reviews_clean.csv', index = False)

In [23]:
print(f"Clean dataset: {df.shape}")
print(f"Date range: {df['timestamp'].min()} -> {df['timestamp'].max()}")
print(df[['timestamp', 'asin', 'text']].head(5))

Clean dataset: (39660, 7)
Date range: 2000-10-07 14:21:50 -> 2023-03-18 02:35:08.237000
            timestamp        asin  \
0 2000-10-07 14:21:50  B00004U8G0   
1 2001-08-11 00:26:51  B00004WGNF   
2 2001-12-06 18:30:01  B00003G1RG   
3 2001-12-12 01:46:38  B00000J4BC   
4 2001-12-12 20:33:53  B00004WGNF   

                                                text  
0  if you love listening to motivational tapes wa...  
1  im just an amateur when it comes to photograph...  
2  if you can get only one addon for your digital...  
3  this is an ok unit and if you want the nifty r...  
4  the camera is very easy to use and takes very ...  
